# ISRO BAH 2026 — PS-07: Notebook 6: Kepler DR24 Download (Pre-training Data)

**Purpose**: Download Kepler DR24 TCE catalog (34,032 labeled TCEs) and associated KIC light curves for CNN pre-training.  
**Requirement**: DATA-04 (Phase 1), CLAS-01 (Phase 2)  
**Run on**: Kaggle CPU notebook (no GPU needed for download)  
**Estimated runtime**: 4–6 hours  
**Output**: Kaggle Dataset `kepler-dr24-tces`

---
⚠ **Run this FIRST during prep week** — it's the slowest download (~4–6h).
The Phase 2 CNN pre-training notebook will mount `kepler-dr24-tces` as input.

In [ ]:
%pip install -q "lightkurve>=2.4" "astroquery>=0.4.7" "pyarrow>=14" "requests>=2.31"


In [ ]:
import sys
from pathlib import Path

PIPELINE_PATH = Path('/kaggle/input/isro-bah-pipeline')
if PIPELINE_PATH.exists():
    sys.path.insert(0, str(PIPELINE_PATH))
else:
    CLONE_DIR = Path('/kaggle/working/ISRO-BAH')
    if not CLONE_DIR.exists():
        GITHUB_URL = 'https://github.com/YOUR_USERNAME/ISRO-BAH.git'  # ← UPDATE THIS
        subprocess.run(['git', 'clone', '--depth=1', GITHUB_URL, str(CLONE_DIR)], check=True)
    sys.path.insert(0, str(CLONE_DIR))

import pipeline.config as cfg
KAGGLE_OUT = Path('/kaggle/working')
cfg.OUTPUTS_DIR = KAGGLE_OUT / 'outputs'
cfg.CATALOGUE_DIR = cfg.OUTPUTS_DIR / 'catalogue'
cfg.KEPLER_TCE_PATH = cfg.CATALOGUE_DIR / 'kepler_dr24_tce.csv'

KEPLER_LC_DIR = cfg.OUTPUTS_DIR / 'kepler'
KEPLER_LC_DIR.mkdir(parents=True, exist_ok=True)
cfg.CATALOGUE_DIR.mkdir(parents=True, exist_ok=True)
print('Config ready. Kepler LC dir:', KEPLER_LC_DIR)

In [ ]:
# Step 1: Download TCE catalog
from pipeline.ingest.download_kepler import download_kepler_tce_catalog
import pandas as pd

df_tce = download_kepler_tce_catalog()
print(f'Kepler DR24 TCE catalog: {len(df_tce)} rows')
if 'label_class' in df_tce.columns:
    print(df_tce['label_class'].value_counts())

kic_ids = sorted(df_tce['kepid'].dropna().astype(int).unique().tolist())
print(f'\nUnique KIC IDs: {len(kic_ids)}')

In [ ]:
# Step 2: Test with 10 KICs before full run
from pipeline.ingest.download_kepler import download_kepler_lightcurves

test_counts = download_kepler_lightcurves(kic_ids=kic_ids, limit=10)
print(f'Test download (n=10): {test_counts}')

In [ ]:
# Step 3: MAIN — Download all Kepler LCs (~4–6h)
import time
print(f'Starting Kepler download at {time.strftime("%H:%M:%S")}')
print(f'Target: {len(kic_ids)} unique KIC IDs')

counts = download_kepler_lightcurves(kic_ids=kic_ids, workers=8)

print(f'\n✓ Done at {time.strftime("%H:%M:%S")}')
print(counts)

In [ ]:
# Final summary
n_downloaded = len(list(KEPLER_LC_DIR.glob('*.npz')))
size_gb = sum(f.stat().st_size for f in KEPLER_LC_DIR.glob('*.npz')) / 1e9
print(f'Kepler LCs: {n_downloaded} files | {size_gb:.1f} GB')
print(f'TCE catalog: {len(df_tce)} rows')
print('\nUpload /kaggle/working/outputs/ as Kaggle Dataset: kepler-dr24-tces')
print('This dataset will be mounted in Phase 2 CNN pre-training notebook.')